# Milestone 01 — Data Prep & Baseline
**Project:** Customer Churn Prediction (Telco) &nbsp;|&nbsp; **Task type:** binary classification

How to use this notebook: run it top to bottom after placing the dataset in `data/raw/` (see `README.md`).
Cells marked **✍️ Your turn** are where you write observations from *your* outputs. Nothing in this notebook is pre-filled
with results, so every number you report should come from your own run.

In [ ]:
import sys
from pathlib import Path

# Make the `src/` package importable whether the notebook is launched from notebooks/ or the repo root
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import set_config
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from src.config import ID_COL, RANDOM_STATE, TARGET, TEST_SIZE
from src.data_preparation import clean, load_raw, make_xy, save_processed
from src.evaluation import cross_validate_on_train, evaluate_on, plot_confusion_matrix, plot_roc
from src.preprocessing import (
    build_dummy_baseline,
    build_logistic_baseline,
    build_preprocessor,
    split_column_types,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
set_config(display="diagram")
print("Setup complete. Random seed:", RANDOM_STATE)

## 1. Problem Definition

| Item | Definition |
| --- | --- |
| Selected problem | Customer churn prediction |
| Problem type | **Binary classification** (churn = 1, stays = 0) |
| Business context | A telecom provider wants to identify customers likely to leave so retention offers can be targeted, since keeping a customer is usually cheaper than acquiring a new one |
| ML objective | Learn a function from customer account/service attributes to the probability that the customer churns |
| Target variable | `Churn` (`Yes`/`No` in the raw file, encoded to 1/0 in `y`) |
| Positive class | Churn (the class the business cares about finding) |

**Assumption (technical):** the recorded attributes are known *before* the churn outcome, so they can be used at prediction time.

## 2. Dataset Overview

* **Name:** Telco Customer Churn (IBM sample dataset)
* **Source:** public Kaggle listing "Telco Customer Churn", file `WA_Fn-UseC_-Telco-Customer-Churn.csv` (download instructions in `README.md`)
* **Expected structure (per public documentation, confirm against your file):** one row per customer, an ID column, demographic fields
  (`gender`, `SeniorCitizen`, `Partner`, `Dependents`), account fields (`tenure`, `Contract`, `PaperlessBilling`, `PaymentMethod`,
  `MonthlyCharges`, `TotalCharges`), service fields (`PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`,
  `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) and the target `Churn`.
* **Feature types:** a few numeric (`tenure`, `MonthlyCharges`, `TotalCharges`), the rest categorical/binary.

**✍️ Your turn:** after running the next cell, record the actual number of rows, number of columns, and the feature/target types here.

In [ ]:
raw = load_raw()
print("Rows, columns:", raw.shape)
raw.head()

In [ ]:
raw.info()

## 3. Exploratory Data Analysis
EDA works on a *copy* of the raw data. Nothing here modifies the data used for modelling, and nothing is fitted.

### 3.1 Structure, types and summary statistics

In [ ]:
raw.describe().T

In [ ]:
raw.describe(exclude="number").T

### 3.2 Missing values, duplicates, unique values

In [ ]:
# Count both real NaNs and blank / whitespace-only strings (which isna() does NOT catch)
text_cols = raw.select_dtypes(exclude="number").columns
blank_counts = {c: int((raw[c].astype(str).str.strip() == "").sum()) for c in text_cols}
missing = pd.DataFrame({
    "nan_count": raw.isna().sum(),
    "blank_string_count": pd.Series(blank_counts),
}).fillna(0).astype(int)

issues = missing[(missing > 0).any(axis=1)]
print("Columns with NaN or blank values:", len(issues))
issues

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
print("Duplicate customer IDs:", raw[ID_COL].duplicated().sum())
raw.nunique().sort_values()

### 3.3 Investigating `TotalCharges`
A numeric column should not load as text. If it did in your run, find out why before converting it.

In [ ]:
print("TotalCharges dtype:", raw["TotalCharges"].dtype)
tc_numeric = pd.to_numeric(raw["TotalCharges"], errors="coerce")
unparsable = raw[tc_numeric.isna()]
print("Rows where TotalCharges is not a valid number:", len(unparsable))
unparsable[[ID_COL, "tenure", "MonthlyCharges", "TotalCharges", TARGET]]

### 3.4 Target distribution

In [ ]:
counts = raw[TARGET].value_counts()
share = raw[TARGET].value_counts(normalize=True).round(3)
display(pd.DataFrame({"count": counts, "share": share}))

fig, ax = plt.subplots(figsize=(4, 3.5))
sns.countplot(data=raw, x=TARGET, order=["No", "Yes"], ax=ax)
ax.set_title("Target distribution")
plt.show()

### 3.5 Numerical features

In [ ]:
# EDA-only frame: numeric TotalCharges and a 0/1 target flag for grouping
eda = raw.copy()
eda["TotalCharges"] = pd.to_numeric(eda["TotalCharges"], errors="coerce")
eda["churn_flag"] = (eda[TARGET] == "Yes").astype(int)
if pd.api.types.is_numeric_dtype(eda["SeniorCitizen"]):
    eda["SeniorCitizen"] = eda["SeniorCitizen"].map({0: "No", 1: "Yes"})

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col in zip(axes, num_cols):
    sns.histplot(data=eda, x=col, hue=TARGET, bins=30, stat="density", common_norm=False, ax=ax)
    ax.set_title(f"{col} by churn")
plt.tight_layout()
plt.show()

In [ ]:
# Potential outliers: boxplots + IQR rule count
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=eda, x=TARGET, y=col, order=["No", "Yes"], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


def iqr_outlier_count(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


pd.Series({c: iqr_outlier_count(eda[c].dropna()) for c in num_cols}, name="IQR-rule outliers")

### 3.6 Categorical features (distribution and churn rate)

In [ ]:
cat_cols = [c for c in eda.select_dtypes(exclude="number").columns if c not in (ID_COL, TARGET)]
print(len(cat_cols), "categorical features")

overall_rate = eda["churn_flag"].mean()
ncols = 4
nrows = int(np.ceil(len(cat_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 3.8 * nrows))
axes = axes.ravel()
for ax, c in zip(axes, cat_cols):
    g = eda.groupby(c)["churn_flag"].agg(["mean", "size"]).sort_values("mean", ascending=False)
    ax.bar(range(len(g)), g["mean"])
    ax.set_xticks(range(len(g)))
    ax.set_xticklabels([f"{i}\n(n={n})" for i, n in zip(g.index, g["size"])], fontsize=7, rotation=25, ha="right")
    ax.axhline(overall_rate, color="red", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_title(c, fontsize=10)
for ax in axes[len(cat_cols):]:
    ax.axis("off")
fig.suptitle("Churn rate by category (bar labels show group size; dashed line = overall churn rate)", y=1.0)
plt.tight_layout()
plt.show()

### 3.7 Relationships with the target, and redundancy between features

In [ ]:
print("Correlation of numeric features with churn flag:")
display(eda[num_cols + ["churn_flag"]].corr()["churn_flag"].drop("churn_flag").round(3))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(eda[num_cols + ["churn_flag"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation (numeric features + churn flag)")
plt.show()

tenure_bins = pd.cut(eda["tenure"], [-1, 6, 12, 24, 48, 72], labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
display(eda.groupby(tenure_bins, observed=True)["churn_flag"].agg(churn_rate="mean", n="size").round(3))

print("corr(TotalCharges, tenure x MonthlyCharges):",
      round(eda["TotalCharges"].corr(eda["tenure"] * eda["MonthlyCharges"]), 3))
print(f"{ID_COL}: {raw[ID_COL].nunique()} unique values in {len(raw)} rows")

## 4. EDA Findings
**✍️ Your turn.** The rows below are *candidate* findings that are commonly relevant for this kind of dataset. Confirm or reject each one using
your own outputs, fill in the Evidence column with your numbers, and delete or add rows as needed.

| Finding (confirm from your output) | Evidence (fill in) | Why it matters | Action |
| --- | --- | --- | --- |
| Target classes are imbalanced | churn share = ___ % | Accuracy can look good while missing most churners | Stratify the split; report precision, recall, F1, ROC-AUC, PR-AUC and the confusion matrix |
| `TotalCharges` is stored as text with blank/unparsable entries | ___ rows; their tenure = ___ | Text dtype breaks numeric processing; blanks may be brand-new customers who have not been billed | Convert to numeric; set to 0 only where `tenure == 0`; impute anything else inside the pipeline |
| `customerID` is unique for every row | ___ unique of ___ rows | An identifier carries no generalisable signal and can let a model memorise rows | Exclude from `X` |
| Tenure and contract type differ visibly between churners and non-churners | churn rate by tenure bin / contract = ___ | Likely strong signals a baseline should pick up | Keep; no action needed now |
| `TotalCharges` is closely tied to `tenure × MonthlyCharges` | correlation = ___ | Multicollinearity makes linear-model coefficients harder to interpret | Keep for the baseline; revisit in feature engineering |
| "No internet service" / "No phone service" categories repeat information already in `InternetService` / `PhoneService` | ___ | Redundant one-hot columns | Keep for the baseline; consider consolidating later |
| Duplicates / other missing values | ___ | Duplicates can leak between train and test | Handled in cleaning (Section 5) |

## 5. Data Cleaning
All cleaning here is **rule-based** (no statistic is estimated from the data), so it is safe to do before the split.
Anything that learns from data (median imputation, scaling, encoding) happens later inside the pipeline, fitted on training data only.
The logic lives in `src/data_preparation.py::clean`.

| Issue | Decision | Reasoning | Trade-off |
| --- | --- | --- | --- |
| Whitespace / blank strings | Strip text; blanks become missing | Blank strings hide missing values from `isna()` | None significant |
| `TotalCharges` stored as text | Convert to numeric (`errors="coerce"`) | It is a monetary amount | Any non-numeric value silently becomes NaN, so the log reports how many |
| `TotalCharges` missing where `tenure == 0` | Set to 0 | A customer with no tenure has not been billed yet | Assumes this explanation is right; the check below verifies it in your data |
| `TotalCharges` missing elsewhere (if any) | Leave as NaN; median-impute in the pipeline | Imputation statistics must come from training data only | Slight distortion if many are missing |
| `SeniorCitizen` coded 0/1 | Recode to No/Yes | Consistent handling with the other binary flags | None |
| Exact duplicate rows | Drop | Duplicates can appear in both train and test | Would remove genuine repeat rows, but exact matches including the ID are unlikely to be genuine |
| Missing/invalid target | Drop rows | Unusable for supervised learning | Loses those rows |
| Outliers | **Not** removed | Charges are bounded, plausible business values; a linear baseline with scaling tolerates them | Revisit if the boxplots show impossible values |

In [ ]:
clean_df, cleaning_log = clean(raw)
display(pd.Series(cleaning_log, name="cleaning log").to_frame())

# Verify the assumption behind the tenure == 0 rule with YOUR data
was_bad = pd.to_numeric(raw["TotalCharges"], errors="coerce").isna()
print("Rows with non-numeric TotalCharges in raw data:", int(was_bad.sum()))
print("...of which have tenure == 0:", int((raw.loc[was_bad, "tenure"] == 0).sum()))
print("Missing values left after cleaning:", int(clean_df.isna().sum().sum()))
print("Shape raw -> clean:", raw.shape, "->", clean_df.shape)
clean_df.dtypes

In [ ]:
out_path = save_processed(clean_df)
print("Saved cleaned data to", out_path)

## 6. Feature & Target Preparation
* `y` = 1 if `Churn == "Yes"` else 0. This matches the business question ("will this customer leave?") and makes churn the positive class.
* `X` = every column except `Churn` and `customerID`.

**Target-leakage review (reasoning):** every remaining feature must be known *before* the customer churns. The account and service attributes
qualify under the dataset's documented meaning, but confirm this against the column descriptions of your file. Columns such as `Churn Label`,
`Churn Score` or `Churn Reason` (present in some expanded versions) would leak the outcome and are dropped by `make_xy` if found.
A data-based leakage screen on the training set follows in Section 7.

In [ ]:
X, y = make_xy(clean_df)
print("X:", X.shape, "| y:", y.shape)
print(y.value_counts().rename({0: "stayed", 1: "churned"}))
X.columns.tolist()

## 7. Train/Test Split
* **80/20 split, stratified on `y`**, with a fixed seed. Stratification keeps the churn share the same in both sets, which matters for an imbalanced target and makes the test metrics comparable to the training ones.
* The split happens **before** any imputation, scaling or encoding is fitted.
* The **test set is used once**, at the end (Section 10). All model comparison inside this milestone uses cross-validation on the training set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
pd.DataFrame(
    {"rows": [len(y_train), len(y_test)], "churn_rate": [y_train.mean(), y_test.mean()]},
    index=["train", "test"],
).round(3)

In [ ]:
# Leakage screen on TRAINING data only (a diagnostic; it must not be used to tune features on the test set).
# Suspiciously "perfect" features can indicate information that would not exist at prediction time.
train_df = X_train.assign(churn=y_train)
pure = []
for c in X_train.select_dtypes(exclude="number").columns:
    g = train_df.groupby(c)["churn"].agg(["mean", "size"])
    if ((g["size"] >= 30) & ((g["mean"] == 0) | (g["mean"] == 1))).any():
        pure.append(c)
print("Categorical features with a category (n >= 30) that is 100% or 0% churn:", pure)

num_train = X_train.select_dtypes(include="number")
single_feature_auc = {}
for c in num_train.columns:
    a = roc_auc_score(y_train, num_train[c].fillna(num_train[c].median()))
    single_feature_auc[c] = max(a, 1 - a)
print("Single-feature ROC-AUC (numeric):", {k: round(v, 3) for k, v in single_feature_auc.items()})

**✍️ Your turn:** an empty list and single-feature AUCs well below ~0.95 support (but do not prove) the absence of leakage. If anything is flagged, investigate what that column means before keeping it.

## 8. Preprocessing Pipeline
Implemented in `src/preprocessing.py` as a `ColumnTransformer`:

| Feature group | Steps |
| --- | --- |
| Numeric | median imputation → standard scaling |
| Categorical | most-frequent imputation → one-hot encoding (`handle_unknown="ignore"`) |

**How leakage is avoided:** the transformer is *unfitted* until it is part of a `Pipeline` that is fitted on training data. Medians, means/standard
deviations and the category lists are learned from the training rows only. During cross-validation the whole pipeline is re-fitted inside each fold,
and at test time the already-fitted pipeline only *transforms*.

In [ ]:
numeric_cols, categorical_cols = split_column_types(X_train)
print("Numeric:", numeric_cols)
print("Categorical:", len(categorical_cols), "columns")

preprocessor = build_preprocessor(X_train)
preprocessor

In [ ]:
# Sanity check on a throw-away copy: fit on TRAIN only, then transform TRAIN and TEST
check = build_preprocessor(X_train).fit(X_train)
print("Transformed train shape:", check.transform(X_train).shape)
print("Transformed test shape :", check.transform(X_test).shape)
print("Number of output features:", len(check.get_feature_names_out()))

## 9. Baseline Model
* **Reference floor — `DummyClassifier` (class prior):** always predicts the majority class. Any real model must beat it; it also shows how misleading accuracy is under imbalance.
* **Baseline model — Logistic Regression** (default regularisation, no class weights, no tuning): simple, fast, well-understood, works with a standardised + one-hot feature matrix, and
  outputs probabilities that support ROC-AUC. Its coefficients are also interpretable.

Deliberately **not** done here: hyperparameter tuning, class weighting, threshold tuning, other model families (reserved for later milestones).

Both models are first compared with stratified 5-fold cross-validation on the **training set only**.

In [ ]:
dummy = build_dummy_baseline(X_train)
logreg = build_logistic_baseline(X_train)

cv_dummy = cross_validate_on_train(dummy, X_train, y_train)
cv_logreg = cross_validate_on_train(logreg, X_train, y_train)

print("Cross-validation on training data - Dummy (majority class)")
display(cv_dummy.round(3))
print("Cross-validation on training data - Logistic Regression")
display(cv_logreg.round(3))

## 10. Baseline Evaluation
Now the models are fitted on the full training set and evaluated **once** on the held-out test set. Threshold = 0.5 (default); it is not tuned in this milestone.

What each metric means for churn:

| Metric | Meaning in this project |
| --- | --- |
| Accuracy | Share of all customers classified correctly. Misleading with imbalance: predicting "stays" for everyone already scores high |
| Precision | Of customers flagged as likely churners, the share who really churn (low precision = wasted retention offers) |
| Recall | Of customers who really churn, the share we catch (low recall = churners we miss and lose) |
| F1-score | Harmonic mean of precision and recall at the 0.5 threshold |
| ROC-AUC | Probability that a random churner is ranked above a random non-churner; threshold-independent ranking quality (0.5 = chance) |
| PR-AUC | Area under the precision-recall curve; more informative than ROC-AUC when the positive class is the minority |
| Confusion matrix | Counts of true/false positives and negatives, showing *which* mistakes are made |

In [ ]:
dummy.fit(X_train, y_train)
logreg.fit(X_train, y_train)

dummy_metrics, _, _ = evaluate_on(dummy, X_test, y_test)
test_metrics, y_pred, y_proba = evaluate_on(logreg, X_test, y_test)

results = pd.DataFrame({"Dummy (majority)": dummy_metrics, "Logistic Regression": test_metrics}).round(3)
results.index.name = "Metric (test set)"
results

In [ ]:
fig1 = plot_confusion_matrix(y_test, y_pred, "Baseline: confusion matrix (test)")
fig2 = plot_roc(y_test, y_proba, "Baseline: ROC curve (test)")
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"], zero_division=0))

## 11. Baseline Interpretation
The coefficients below come from the model fitted on the **training** set. They are descriptive only: features are standardised/one-hot encoded, and correlated
features (e.g. `TotalCharges` with `tenure`, or the redundant "No internet service" categories) share credit, so treat the ranking as a rough guide, not causal evidence.

In [ ]:
feature_names = logreg.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(logreg.named_steps["model"].coef_[0], index=feature_names).sort_values()
top = pd.concat([coefs.head(10), coefs.tail(10)])

fig, ax = plt.subplots(figsize=(8, 6))
top.plot.barh(ax=ax, color=["tab:blue" if v < 0 else "tab:red" for v in top])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Largest logistic-regression coefficients (negative = lower churn odds)")
plt.tight_layout()
plt.show()

**✍️ Your turn — write your interpretation from your own numbers.** Prompts to answer:

1. How does the logistic regression compare with the dummy model on each metric? Is the gain in ROC-AUC/PR-AUC meaningful, and is accuracy alone a fair way to judge it?
2. Compare precision and recall. Which kind of error (missed churners vs. wasted offers) dominates, and what would each cost the business?
3. Do cross-validation scores (train) and test scores agree? A big gap would point to variance or a lucky/unlucky split.
4. Is there evidence of **underfitting** (weak performance on both train-CV and test)? A linear model may be too simple for interactions between contract, tenure and services.
5. Do the largest coefficients make business sense, or do any look suspicious (possible leakage)?

## 12. Risks & Limitations
* **Data:** a public, IBM-generated sample dataset — it may not reflect real telecom churn behaviour, and the exact rows/columns depend on the version you downloaded.
* **Snapshot data:** no timestamps, so a time-based split (train on the past, test on the future) is impossible; the model cannot be checked for drift.
* **Potential leakage:** no obvious post-churn columns in the standard 21-column file, but this is a judgement based on documentation; the expanded versions do contain leaking columns.
* **Class imbalance:** the minority (churn) class limits how well recall and precision can both be high; accuracy is not a reliable headline metric.
* **Sampling:** unknown how customers were selected; results may not transfer to other regions, plans or time periods.
* **Modelling:** a single, untuned linear model at a default 0.5 threshold; no interaction terms; no class weighting.
* **Evaluation:** one 80/20 split gives a single point estimate; with ~1,400 test rows (if you use the standard file) the metrics carry noticeable uncertainty. Cross-validation on train helps quantify that.

## 13. Next-Milestone Direction
Ideas to consider later (not done here): class weighting or resampling; threshold selection based on retention-offer cost; regularisation-strength tuning; tree-based models
(random forest, gradient boosting) to capture interactions; feature engineering (e.g. number of add-on services, tenure buckets, charge-per-month ratios); probability calibration;
cost-sensitive evaluation; and more robust evaluation (repeated CV, confidence intervals).

## 14–15. Repository Structure & README
See the repository root: `README.md` documents the structure, setup, how to reproduce the baseline (`python -m src.train_baseline`), and the results table to fill from `reports/baseline_metrics.json`.

## 16. Milestone Completion Checklist
- [ ] Problem correctly identified as classification
- [ ] Dataset selected and documented (Sections 1–2)
- [ ] EDA completed (Section 3)
- [ ] Missing values investigated
- [ ] Duplicates investigated
- [ ] Data types checked
- [ ] Target distribution investigated
- [ ] Data cleaned appropriately (Section 5)
- [ ] Target leakage checked (Sections 6–7)
- [ ] `X` and `y` correctly defined
- [ ] Train/test split implemented
- [ ] Preprocessing pipeline implemented
- [ ] Baseline model trained
- [ ] Appropriate metrics selected
- [ ] Baseline evaluated (test set used once)
- [ ] Results interpreted in your own words (Sections 4 and 11 filled in)
- [ ] Notebook organized and re-run top to bottom
- [ ] Repository organized
- [ ] README prepared with your real results
- [ ] Work is reproducible